In [1]:
pip install pennylane 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 683.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 16.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 36.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 47.9 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [6]:
"""
QCN vs Baselines — Barren Plateau Benchmark
Saves ALL experiment results to CSV. No plots.

KEPT:    Exp 1 (grad var vs qubits), Exp 4 (grad var vs depth — redesigned),
         Exp 7 (param efficiency), Exp 8 (cap ablation)
NEW:     Exp 2 (cost landscape 1D+2D), Exp 3 (gradient histogram),
         Exp 5 (init sensitivity)
REMOVED: Exp 2-old(3D landscape), Exp 3-old(convergence),
         Exp 5-old(log-log), Exp 6(expr vs train), B1-Local
"""

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from scipy.optimize import curve_fit
import warnings, os, time, csv
warnings.filterwarnings('ignore')

np.random.seed(42)
T0 = time.time()

# ═══════════════════════════════════════════════════════════════
# Config
# ═══════════════════════════════════════════════════════════════
N_LAYERS     = 2
N_LAYERS_B   = 2
N_SEEDS      = 30
N_TRIALS     = 3
CAP_SIZE     = 2

QUBIT_RANGES = [6, 8, 10, 12, 14, 16, 18, 20]
FIXED_N      = 12
DEPTH_RANGE  = [1, 2, 3, 4, 5, 10, 15, 20, 25]
CAP_SIZES    = [2, 3, 4, 5, 6, 9]

# Exp 4 redesign — per-model subplot, lines = qubit counts
EXP4_QUBIT_RANGES = [2, 4, 6, 8, 10, 12, 14, 18]
EXP4_DEPTH_RANGE  = [1, 2, 3, 4, 5, 10, 15, 20, 25]

# Exp 2 — Cost Landscape
LANDSCAPE_1D_POINTS = 60
LANDSCAPE_2D_GRID   = 25

# Exp 3 — Gradient Histogram
GRAD_HIST_SEEDS    = 100
GRAD_HIST_N_PARAMS = 6

# Exp 5 — Init Sensitivity
INIT_SEEDS      = 25
INIT_STRATEGIES = ['uniform', 'narrow', 'identity', 'normal']

OUTPUT_DIR = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════════
# Circuit Builders  (B1-Local removed)
# ═══════════════════════════════════════════════════════════════

def build_qcn(n, dev, depth=N_LAYERS, cap_size=CAP_SIZE):
    n_caps = n // cap_size
    n_p    = depth * n * 4

    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(depth, n, 4)
        for i in range(n):
            qml.Hadamard(wires=i)
        for layer in range(depth):
            for i in range(n):
                qml.RY(x[i] * p[layer, i, 3], wires=i)
            for c in range(n_caps):
                w = list(range(c * cap_size, (c + 1) * cap_size))
                for j, wire in enumerate(w):
                    qml.Rot(p[layer, c*cap_size+j, 0],
                            p[layer, c*cap_size+j, 1],
                            p[layer, c*cap_size+j, 2], wires=wire)
            if layer % 2 == 0:
                for c in range(n_caps):
                    w = list(range(c * cap_size, (c + 1) * cap_size))
                    for j in range(len(w) - 1):
                        qml.CNOT(wires=[w[j], w[j+1]])
                    if len(w) > 2:
                        qml.CNOT(wires=[w[-1], w[0]])
            if layer % 2 == 0:
                offset = (layer // 2) % 2
                for c in range(offset, n_caps - 1, 2):
                    boundary_a = (c + 1) * cap_size - 1
                    boundary_b = (c + 1) * cap_size
                    qml.CNOT(wires=[boundary_a, boundary_b])
        return qml.math.stack([
            qml.expval(qml.PauliZ(c * cap_size)) for c in range(n_caps)
        ])
    return circuit, n_p


def build_b2_qmps(n, dev, depth=N_LAYERS_B, cap_size=CAP_SIZE):
    n_p = depth * (n - 1) * 8
    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(depth, n-1, 8)
        for i in range(n): qml.RY(x[i], wires=i)
        for layer in range(depth):
            for i in range(n - 1):
                qml.Rot(p[layer, i, 0], p[layer, i, 1], p[layer, i, 2], wires=i)
                qml.Rot(p[layer, i, 4], p[layer, i, 5], p[layer, i, 6], wires=i+1)
                qml.CNOT(wires=[i, i+1])
                qml.RZ(p[layer, i, 3], wires=i)
                qml.RZ(p[layer, i, 7], wires=i+1)
        return qml.expval(qml.PauliZ(n // 2))
    return circuit, n_p


def build_b3_layerwise(n, dev, depth=N_LAYERS_B, cap_size=CAP_SIZE):
    n_p = depth * n * 2
    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(depth, n, 2)
        for i in range(n): qml.RY(x[i], wires=i)
        for layer in range(depth):
            for i in range(n): qml.RY(p[layer, i, 0], wires=i)
            if layer % 2 == 0:
                for i in range(0, n - 1, 2): qml.CNOT(wires=[i, i + 1])
            else:
                for i in range(1, n - 1, 2): qml.CNOT(wires=[i, i + 1])
            for i in range(n): qml.RZ(p[layer, i, 1], wires=i)
        return qml.math.stack([qml.expval(qml.PauliZ(i)) for i in range(n)])
    return circuit, n_p


def build_b5_resqnet(n, dev, depth=N_LAYERS_B, cap_size=CAP_SIZE):
    n_p = depth * n * 4
    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(depth, n, 4)
        for i in range(n): qml.RY(x[i], wires=i)
        for block in range(depth):
            for i in range(n):
                qml.Rot(p[block, i, 0], p[block, i, 1], p[block, i, 2], wires=i)
            for i in range(n - 1): qml.CNOT(wires=[i, i + 1])
            qml.CNOT(wires=[n - 1, 0])
            for i in range(n): qml.RY(p[block, i, 3] * x[i], wires=i)
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(n - 1))
    return circuit, n_p


def build_b6_global(n, dev, depth=N_LAYERS_B, cap_size=CAP_SIZE):
    n_p = depth * n * 2
    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(depth, n, 2)
        for i in range(n): qml.RY(x[i], wires=i)
        for layer in range(depth):
            for i in range(n): qml.RY(p[layer, i, 0], wires=i)
            for i in range(n - 1): qml.CNOT(wires=[i, i+1])
            qml.CNOT(wires=[0, n-1])
            for i in range(n): qml.RZ(p[layer, i, 1], wires=i)
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(n - 1))
    return circuit, n_p


def build_b7_glob96(n, dev, depth=N_LAYERS_B, cap_size=CAP_SIZE):
    base_nl       = max(1, 96 // (n * 2))
    actual_layers = max(1, round(base_nl * depth / N_LAYERS_B))
    n_p           = actual_layers * n * 2
    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        p = params.reshape(actual_layers, n, 2)
        for i in range(n): qml.RY(x[i], wires=i)
        for layer in range(actual_layers):
            for i in range(n): qml.RY(p[layer, i, 0], wires=i)
            for i in range(n - 1): qml.CNOT(wires=[i, i+1])
            qml.CNOT(wires=[0, n-1])
            for i in range(n): qml.RZ(p[layer, i, 1], wires=i)
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(n - 1))
    return circuit, n_p


BUILDERS = {
    'QCN':        build_qcn,
    'B2-qMPS':    build_b2_qmps,
    'B3-Layer':   build_b3_layerwise,
    'B5-ResQNet': build_b5_resqnet,
    'B6-Global':  build_b6_global,
    'B7-Glob96':  build_b7_glob96,
}


# ═══════════════════════════════════════════════════════════════
# Helpers
# ═══════════════════════════════════════════════════════════════

def cost_from_out(out):
    return pnp.mean(out**2) if hasattr(out, '__len__') else out**2

def get_depth(name):
    return N_LAYERS if name == 'QCN' else N_LAYERS_B

def get_param_idx(name, n_p, n, depth):
    if name == 'QCN':
        k = (depth - 1) * n * 4
    elif name in ('B3-Layer',):
        k = (depth - 1) * n * 2
    elif name == 'B2-qMPS':
        k = (depth - 1) * (n - 1) * 8
    elif name == 'B5-ResQNet':
        k = (depth - 1) * n * 4
    else:
        k = max(0, n_p - n * 2)
    return min(k, n_p - 1)

def compute_grad_var_single(name, circ, n_p, n, depth, base_seed=0):
    k = get_param_idx(name, n_p, n, depth)
    grad_samples = []
    for s in range(N_SEEDS):
        seed = base_seed * 1000 + s * 13 + 7
        pnp.random.seed(seed); np.random.seed(seed)
        p = pnp.random.uniform(0, 2*np.pi, n_p, requires_grad=True)
        x = pnp.array(np.random.uniform(0, np.pi, n), requires_grad=False)
        try:
            def cost_fn(pp, c=circ, xx=x):
                return cost_from_out(c(pp, xx))
            g = np.array(qml.grad(cost_fn)(p)).flatten()
            if k < len(g):
                grad_samples.append(float(g[k]))
        except Exception:
            pass
    return float(np.var(grad_samples)) if len(grad_samples) >= 2 else 1e-20

def compute_grad_var(name, circ, n_p, n, depth):
    trial_vars = [
        compute_grad_var_single(name, circ, n_p, n, depth, base_seed=t)
        for t in range(N_TRIALS)
    ]
    return float(np.mean(trial_vars)), float(np.std(trial_vars))

def write_csv(filename, fieldnames, rows):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    kb = os.path.getsize(path) // 1024
    print(f'  ✓  {filename}  ({kb} KB,  {len(rows)} rows)')


# ═══════════════════════════════════════════════════════════════
# EXP 1 — Gradient Variance vs Qubits
# ═══════════════════════════════════════════════════════════════
print('\n[EXP 1] Gradient Variance vs Qubits ...')
rows1 = []

for n in QUBIT_RANGES:
    print(f'  n={n:2d} | ', end='', flush=True)
    dev = qml.device('lightning.qubit', wires=n)
    for name, build in BUILDERS.items():
        try:
            depth_here = get_depth(name)
            circ, n_p  = build(n, dev, depth=depth_here)
            mean, std  = compute_grad_var(name, circ, n_p, n, depth_here)
            rows1.append({'model': name, 'n_qubits': n,
                          'grad_var_mean': mean, 'grad_var_std': std})
            print(f'{name}={mean:.1e} ', end='', flush=True)
        except Exception as e:
            print(f'{name}=ERR ', end='', flush=True)
    print()

write_csv('exp1_grad_var_qubits.csv',
          ['model', 'n_qubits', 'grad_var_mean', 'grad_var_std'], rows1)


# ═══════════════════════════════════════════════════════════════
# EXP 2 — Cost Landscape  (1D sweep + 2D contour grid)
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 2] Cost Landscape (n={FIXED_N}) ...')
rows2_1d = []
rows2_2d = []

theta_1d = np.linspace(0, 2*np.pi, LANDSCAPE_1D_POINTS)
theta_2d = np.linspace(0, 2*np.pi, LANDSCAPE_2D_GRID)
T1, T2   = np.meshgrid(theta_2d, theta_2d)

for name, build in BUILDERS.items():
    print(f'  {name} ...', flush=True)
    np.random.seed(42); pnp.random.seed(42)
    dev = qml.device('lightning.qubit', wires=FIXED_N)
    try:
        depth_here = get_depth(name)
        circ, n_p  = build(FIXED_N, dev, depth=depth_here)
        pb = np.random.uniform(0, 2*np.pi, n_p)
        xv = np.random.uniform(0, np.pi, FIXED_N)

        # ── 1D sweep: θ₀ varied, rest fixed ──
        for theta_val in theta_1d:
            pc = pb.copy(); pc[0] = theta_val
            out  = circ(pnp.array(pc), pnp.array(xv))
            cost = float(cost_from_out(pnp.array(np.array(out))))
            rows2_1d.append({'model': name, 'theta': round(float(theta_val), 6),
                             'cost': cost})

        # ── 2D grid: θ₀, θ₁ varied ──
        for i in range(LANDSCAPE_2D_GRID):
            for j in range(LANDSCAPE_2D_GRID):
                pc = pb.copy(); pc[0] = T1[i,j]; pc[1] = T2[i,j]
                out  = circ(pnp.array(pc), pnp.array(xv))
                cost = float(cost_from_out(pnp.array(np.array(out))))
                rows2_2d.append({'model': name,
                                 'theta1': round(float(T1[i,j]), 6),
                                 'theta2': round(float(T2[i,j]), 6),
                                 'cost': cost})
    except Exception as e:
        print(f'    skip ({e})')

write_csv('exp2_landscape_1d.csv',
          ['model', 'theta', 'cost'], rows2_1d)
write_csv('exp2_landscape_2d.csv',
          ['model', 'theta1', 'theta2', 'cost'], rows2_2d)


# ═══════════════════════════════════════════════════════════════
# EXP 3 — Gradient Concentration Histogram
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 3] Gradient Histogram (n={FIXED_N}) ...')
rows3 = []

for name, build in BUILDERS.items():
    print(f'  {name} ...', flush=True)
    dev = qml.device('lightning.qubit', wires=FIXED_N)
    depth_here = get_depth(name)
    try:
        circ, n_p = build(FIXED_N, dev, depth=depth_here)

        for s in range(GRAD_HIST_SEEDS):
            seed = s * 17 + 3
            pnp.random.seed(seed); np.random.seed(seed)
            p = pnp.random.uniform(0, 2*np.pi, n_p, requires_grad=True)
            x = pnp.array(np.random.uniform(0, np.pi, FIXED_N), requires_grad=False)
            try:
                def cost_fn(pp, c=circ, xx=x):
                    return cost_from_out(c(pp, xx))
                g = np.array(qml.grad(cost_fn)(p)).flatten()
                indices = list(range(min(GRAD_HIST_N_PARAMS, len(g))))
                for idx in indices:
                    rows3.append({'model': name, 'seed': s,
                                  'param_idx': idx,
                                  'gradient': float(g[idx])})
            except Exception:
                pass
    except Exception as e:
        print(f'    skip ({e})')

write_csv('exp3_gradient_histogram.csv',
          ['model', 'seed', 'param_idx', 'gradient'], rows3)


# ═══════════════════════════════════════════════════════════════
# EXP 4 — Gradient Variance vs Depth
#          One subplot per model; lines = different qubit counts
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 4] Gradient Variance vs Depth (per model × qubit count) ...')
rows4 = []

for name, build in BUILDERS.items():
    print(f'  [{name}]')
    for n in EXP4_QUBIT_RANGES:
        print(f'    n={n:2d} | ', end='', flush=True)
        for depth in EXP4_DEPTH_RANGE:
            dev = qml.device('lightning.qubit', wires=n)
            try:
                circ, n_p = build(n, dev, depth=depth)
                mean, std = compute_grad_var(name, circ, n_p, n, depth)
                rows4.append({'model': name, 'n_qubits': n,
                              'depth': depth,
                              'grad_var_mean': mean,
                              'grad_var_std': std})
                print(f'd{depth}={mean:.1e} ', end='', flush=True)
            except Exception:
                print(f'd{depth}=ERR ', end='', flush=True)
        print()

write_csv('exp4_grad_var_depth.csv',
          ['model', 'n_qubits', 'depth', 'grad_var_mean', 'grad_var_std'],
          rows4)


# ═══════════════════════════════════════════════════════════════
# EXP 5 — Initialization Sensitivity
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 5] Initialization Sensitivity (n={FIXED_N}) ...')
rows5 = []

def make_init(strategy, n_p, seed):
    np.random.seed(seed)
    if strategy == 'uniform':
        return np.random.uniform(0, 2*np.pi, n_p)
    elif strategy == 'narrow':
        return np.random.normal(np.pi, 0.1, n_p)
    elif strategy == 'identity':
        return np.zeros(n_p)
    elif strategy == 'normal':
        return np.random.normal(0, 1, n_p)
    return np.random.uniform(0, 2*np.pi, n_p)

for name, build in BUILDERS.items():
    print(f'  {name} ...', flush=True)
    dev = qml.device('lightning.qubit', wires=FIXED_N)
    depth_here = get_depth(name)
    try:
        circ, n_p = build(FIXED_N, dev, depth=depth_here)
        k = get_param_idx(name, n_p, FIXED_N, depth_here)
        for strategy in INIT_STRATEGIES:
            for s in range(INIT_SEEDS):
                seed   = s * 31 + 5
                p_init = make_init(strategy, n_p, seed)
                xseed  = seed + 1000
                np.random.seed(xseed)
                x = np.random.uniform(0, np.pi, FIXED_N)
                p  = pnp.array(p_init, requires_grad=True)
                xv = pnp.array(x, requires_grad=False)
                try:
                    def cost_fn(pp, c=circ, xx=xv):
                        return cost_from_out(c(pp, xx))
                    g = np.array(qml.grad(cost_fn)(p)).flatten()
                    if k < len(g):
                        rows5.append({'model': name,
                                      'init_strategy': strategy,
                                      'seed': s,
                                      'gradient': float(g[k])})
                except Exception:
                    pass
    except Exception as e:
        print(f'    skip ({e})')

write_csv('exp5_init_sensitivity.csv',
          ['model', 'init_strategy', 'seed', 'gradient'], rows5)


# ═══════════════════════════════════════════════════════════════
# EXP 7 — Parameter Efficiency
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 7] Parameter Efficiency (n={FIXED_N}) ...')
rows7 = []

gv_fixed = {}
for row in rows1:
    if row['n_qubits'] == FIXED_N:
        gv_fixed[row['model']] = row['grad_var_mean']

for name, build in BUILDERS.items():
    dev = qml.device('lightning.qubit', wires=FIXED_N)
    try:
        _, n_p   = build(FIXED_N, dev, depth=get_depth(name))
        gv_val   = gv_fixed.get(name, 1e-20)
        efficiency = gv_val / max(n_p, 1)
        rows7.append({'model': name, 'n_params': n_p,
                      'grad_var_at_n12': gv_val,
                      'param_efficiency': efficiency})
    except Exception as e:
        print(f'  {name} skip ({e})')

rows7.sort(key=lambda r: r['param_efficiency'], reverse=True)
write_csv('exp7_param_efficiency.csv',
          ['model', 'n_params', 'grad_var_at_n12', 'param_efficiency'],
          rows7)


# ═══════════════════════════════════════════════════════════════
# EXP 8 — Cap Size Ablation  (QCN only)
# ═══════════════════════════════════════════════════════════════
print(f'\n[EXP 8] Cap Size Ablation (QCN) ...')
rows8 = []

for cs in CAP_SIZES:
    print(f'  cap_size={cs}', flush=True)
    for n in QUBIT_RANGES:
        if n % cs != 0:
            continue
        dev = qml.device('lightning.qubit', wires=n)
        try:
            circ, n_p = build_qcn(n, dev, depth=N_LAYERS, cap_size=cs)
            mean, std = compute_grad_var('QCN', circ, n_p, n, N_LAYERS)
            rows8.append({'cap_size': cs, 'n_qubits': n,
                          'grad_var_mean': mean, 'grad_var_std': std})
        except Exception:
            pass

write_csv('exp8_cap_ablation.csv',
          ['cap_size', 'n_qubits', 'grad_var_mean', 'grad_var_std'],
          rows8)


# ═══════════════════════════════════════════════════════════════
# DONE
# ═══════════════════════════════════════════════════════════════
elapsed = time.time() - T0
print(f'\n{"="*55}')
print(f'  All CSVs saved to:\n  {OUTPUT_DIR}')
print(f'  Total runtime: {elapsed:.0f}s  ({elapsed/60:.1f} min)')
print(f'{"="*55}')
print('\n  Files generated:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    if fname.endswith('.csv'):
        kb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) // 1024
        print(f'    {fname:<45} {kb:>5} KB')


[EXP 1] Gradient Variance vs Qubits ...
  n= 6 | QCN=3.5e-03 B2-qMPS=1.1e-03 B3-Layer=6.4e-03 B5-ResQNet=1.5e-03 B6-Global=1.2e-03 B7-Glob96=1.2e-03 
  n= 8 | QCN=2.9e-03 B2-qMPS=4.2e-04 B3-Layer=3.2e-03 B5-ResQNet=3.2e-04 B6-Global=8.1e-06 B7-Glob96=9.6e-05 
  n=10 | QCN=1.6e-03 B2-qMPS=6.6e-05 B3-Layer=2.4e-03 

KeyboardInterrupt: 

In [12]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv, os

# ── Paths ─────────────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/input/datasets/tasfiazamansamiha/barren-pleatau/baseline experiments'
CSV_PATH   = os.path.join(OUTPUT_DIR, 'exp1_grad_var_qubits.csv')
OUT_PATH   = '/kaggle/working/exp1_grad_var_qubits_plot.png'

# ── Read CSV ──────────────────────────────────────────────────
data = {}
with open(CSV_PATH, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        model = row['model']
        if model not in data:
            data[model] = {'n': [], 'mean': []}
        data[model]['n'].append(int(row['n_qubits']))
        data[model]['mean'].append(float(row['grad_var_mean']))

print("Models found in CSV:", list(data.keys()))

# Sort by qubit count
for model in data:
    pairs = sorted(zip(data[model]['n'], data[model]['mean']))
    ns, means = zip(*pairs)
    data[model] = {'n': list(ns), 'mean': list(means)}

# ── Style map — exact colors from reference image ─────────────
# রঙের কোডগুলো মূল ছবির সাথে নিখুঁতভাবে ম্যাচ করানো হয়েছে
STYLE_MAP = {
    'QCN':        {'color': '#2ca02c', 'label': 'QCN',      'lw': 2.5, 'ms': 8, 'zorder': 5},
    
    'B2-qMPS':    {'color': '#9467bd', 'label': 'qMPS',     'lw': 2.5, 'ms': 8, 'zorder': 4},
    'B3-Layer':   {'color': '#ff7f0e', 'label': 'Layer',    'lw': 2.5, 'ms': 8, 'zorder': 4},
    'B5-ResQNet': {'color': '#d62728', 'label': 'ResQNet',  'lw': 2.5, 'ms': 8, 'zorder': 5},
    'B6-Global':  {'color': '#8c564b', 'label': 'Global',   'lw': 2.5, 'ms': 8, 'zorder': 3},
    'B7-Glob96':  {'color': '#e377c2', 'label': 'Global96', 'lw': 2.5, 'ms': 8, 'zorder': 3},
    
    # Kaggle alternative names
    'Local':      {'color': '#1f77b4', 'label': 'Local',    'lw': 2.5, 'ms': 8, 'zorder': 6},
    'qMPS':       {'color': '#9467bd', 'label': 'qMPS',     'lw': 2.5, 'ms': 8, 'zorder': 4},
    'Layer':      {'color': '#ff7f0e', 'label': 'Layer',    'lw': 2.5, 'ms': 8, 'zorder': 4},
    'ResQNet':    {'color': '#d62728', 'label': 'ResQNet',  'lw': 2.5, 'ms': 8, 'zorder': 5},
    'Global':     {'color': '#8c564b', 'label': 'Global',   'lw': 2.5, 'ms': 8, 'zorder': 3},
    'Glob96':     {'color': '#e377c2', 'label': 'Global96', 'lw': 2.5, 'ms': 8, 'zorder': 3},
}

# ── Figure style — matching reference exactly ─────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         12,
    'axes.titlesize':    16,
    'axes.labelsize':    15,
    'legend.fontsize':   12,
    'xtick.labelsize':   12,
    'ytick.labelsize':   12,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.edgecolor':    'black',     # বর্ডার একদম স্পষ্ট কালো হবে
    'axes.linewidth':    1.2,
    'xtick.direction':   'in',        # টিক্স ভেতরের দিকে থাকবে
    'ytick.direction':   'in',
    'xtick.major.size':  5,
    'ytick.major.size':  5,
})

fig, ax = plt.subplots(figsize=(8.5, 6.5))

# ── Plot each model ───────────────────────────────────────────
plotted = set()
for model, d in data.items():
    style = STYLE_MAP.get(model)
    if style is None:
        print(f"  Warning: '{model}' not in STYLE_MAP — skipping")
        continue

    if style['label'] in plotted:
        continue
    plotted.add(style['label'])

    ns    = np.array(d['n'])
    means = np.array(d['mean'])

    ax.plot(
        ns, means,
        marker='o',
        linestyle='-',
        color=style['color'],
        lw=style['lw'],
        ms=style['ms'],
        label=style['label'],
        zorder=style['zorder']
    )

# ── Axes Settings ─────────────────────────────────────────────
ax.set_yscale('log')
ax.set_xlabel('Number of Qubits', labelpad=10)
ax.set_ylabel('Variance of Gradients', labelpad=10)
ax.set_title('Gradient Variance vs. Number of Qubits', pad=12)

# X-axis ticks (6, 8, 10, 12, 14, 16)
all_ns = sorted(set(n for d in data.values() for n in d['n']))
ax.set_xticks(all_ns)
ax.set_xticklabels([str(n) for n in all_ns])

# গ্রিড স্টাইল (শুধুমাত্র মেজর গ্রিড থাকবে এবং হালকা ড্যাশড হবে)
ax.grid(True, which='major', color='#e0e0e0', linestyle='--', linewidth=0.8)
ax.grid(False, which='minor') # মাইনর গ্রিড বন্ধ

# চারপাশের বর্ডার অন রাখা (যা মূল ছবিতে আছে)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# ── Legend — alphabetical, lower left ────────────────────────
handles, labels = ax.get_legend_handles_labels()
sorted_pairs = sorted(zip(labels, handles), key=lambda x: x[0])
sorted_labels, sorted_handles = zip(*sorted_pairs)

ax.legend(
    sorted_handles, sorted_labels,
    loc='lower left',
    framealpha=1.0,
    edgecolor='#cccccc',
    facecolor='white',
    borderpad=0.6,
    labelspacing=0.5,
    handletextpad=0.6
)

plt.tight_layout()

# ── Save Plot ─────────────────────────────────────────────────
plt.savefig(OUT_PATH, dpi=300, bbox_inches='tight')
plt.close()

kb = os.path.getsize(OUT_PATH) // 1024
print(f'✓ Saved successfully: {OUT_PATH} ({kb} KB)')

Models found in CSV: ['QCN', 'B2-qMPS', 'B3-Layer', 'B5-ResQNet', 'B6-Global', 'B7-Glob96']
✓ Saved successfully: /kaggle/working/exp1_grad_var_qubits_plot.png (292 KB)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ── Paths ─────────────────────────────────────────────────────
CSV_PATH = r"C:\Users\User\Downloads\baseline experiments\exp2_landscape_1d.csv"
OUT_PATH = r"C:\Users\User\Downloads\baseline experiments\exp2_landscape_1d_plot.png"

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ── Read CSV ──────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

print("Models found in CSV:", df['model'].unique())

# ── Figure Style ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         12,
    'axes.titlesize':    16,
    'axes.labelsize':    14,
    'xtick.labelsize':   12,
    'ytick.labelsize':   12,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.edgecolor':    'black',
    'axes.linewidth':    1.2,
    'xtick.direction':   'in',
    'ytick.direction':   'in',
    'xtick.major.size':  5,
    'ytick.major.size':  5,
})

fig, ax = plt.subplots(figsize=(8.5, 6.5))

# ── Your Exact Style Map ──────────────────────────────────────
STYLE_MAP = {
    'QCN':        {'color': '#2ca02c', 'label': 'QCapsule',  'lw': 2.5, 'ms': 8, 'zorder': 5},
    'B1-Local':   {'color': '#1f77b4', 'label': 'Local',     'lw': 2.5, 'ms': 8, 'zorder': 6},
    'B2-qMPS':    {'color': '#9467bd', 'label': 'qMPS',      'lw': 2.5, 'ms': 8, 'zorder': 4},
    'B3-Layer':   {'color': '#ff7f0e', 'label': 'Layer',     'lw': 2.5, 'ms': 8, 'zorder': 4},
    'B5-ResQNet': {'color': '#d62728', 'label': 'ResQNet',   'lw': 2.5, 'ms': 8, 'zorder': 5},
    'B6-Global':  {'color': '#8c564b', 'label': 'Global',    'lw': 2.5, 'ms': 8, 'zorder': 3},
    'B7-Glob96':  {'color': '#e377c2', 'label': 'Global96',  'lw': 2.5, 'ms': 8, 'zorder': 3},
    
    # Kaggle alternative names
    'Local':      {'color': '#1f77b4', 'label': 'Local',     'lw': 2.5, 'ms': 8, 'zorder': 6},
    'qMPS':       {'color': '#9467bd', 'label': 'qMPS',      'lw': 2.5, 'ms': 8, 'zorder': 4},
    'Layer':      {'color': '#ff7f0e', 'label': 'Layer',     'lw': 2.5, 'ms': 8, 'zorder': 4},
    'ResQNet':    {'color': '#d62728', 'label': 'ResQNet',   'lw': 2.5, 'ms': 8, 'zorder': 5},
    'Global':     {'color': '#8c564b', 'label': 'Global',    'lw': 2.5, 'ms': 8, 'zorder': 3},
    'Glob96':     {'color': '#e377c2', 'label': 'Global96',  'lw': 2.5, 'ms': 8, 'zorder': 3},
}

# ── Plot Data ─────────────────────────────────────────────────
plotted_labels = set()

for model_raw_name in df['model'].unique():
    style = STYLE_MAP.get(model_raw_name)
    if style is None:
        print(f"  Warning: '{model_raw_name}' not in STYLE_MAP — skipping")
        continue
        
    if style['label'] in plotted_labels:
        continue
    plotted_labels.add(style['label'])
    
    group = df[df['model'] == model_raw_name].sort_values('theta')
    
    ax.plot(
        group['theta'], 
        group['cost'], 
        label=style['label'], 
        color=style['color'], 
        linewidth=style['lw'],
        linestyle='-',
        zorder=style['zorder']
    )

# ── Axes Settings (লগ স্কেল অ্যাক্টিভেট করা হয়েছে) ────────────────
ax.set_yscale('log') # এই লাইনটি ছোট মানগুলোকে ০ থেকে আলাদা করে স্পষ্ট করবে

ax.set_xlabel(r'$\theta$', labelpad=10, fontsize=15)
ax.set_ylabel('Cost Value', labelpad=10)
ax.set_title('1D Cost Landscape', pad=12)

# গ্রিড স্টাইল (মেজর এবং মাইনর দুইটাই অন করা হয়েছে যাতে লগ স্কেলে দেখতে সুবিধা হয়)
ax.grid(True, which='major', color='#cccccc', linestyle='--', linewidth=0.8, zorder=1)
ax.grid(True, which='minor', color='#f0f0f0', linestyle='--', linewidth=0.5, zorder=1)

# চারপাশের স্পষ্ট কালো বর্ডার
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# ── Legend ────────────────────────────────────────────────────
handles, labels = ax.get_legend_handles_labels()
sorted_pairs = sorted(zip(labels, handles), key=lambda x: x[0])
sorted_labels, sorted_handles = zip(*sorted_pairs)

ax.legend(
    sorted_handles, sorted_labels,
    loc='lower left',        
    framealpha=1.0,
    edgecolor='#cccccc',
    facecolor='white',
    borderpad=0.7,
    labelspacing=0.5,
    handlelength=2.2
)

plt.tight_layout()

# ── Save Plot ─────────────────────────────────────────────────
plt.savefig(OUT_PATH, dpi=300, bbox_inches='tight')
plt.close()

print(f'✓ Log-scaled graph successfully saved at:\n  {OUT_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import griddata
import os

# ── Paths ─────────────────────────────────────────────────────
CSV_PATH = r"C:\Users\User\Downloads\baseline experiments\exp2_landscape_2d.csv"
OUT_PATH = r"C:\Users\User\Downloads\baseline experiments\exp2_landscape_2d_plot.png"

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# ── Read CSV ──────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# ১. ডাটা ক্লিনিং: মডেলের নাম থেকে B1, B2 প্রিফিক্স বাদ দেওয়া এবং QCN-কে QCapsule করা
def clean_model_name(name):
    # যদি নামের মধ্যে হাইফেন (-) থাকে, তবে হাইফেনের পরের অংশটুকু নেওয়া হবে (যেমন: B2-qMPS -> qMPS)
    if '-' in name:
        name = name.split('-', 1)[1]
    
    # QCN নাম পরিবর্তন করে QCapsule করা
    if name == 'QCN':
        return 'QCapsule'
    return name

df['model'] = df['model'].apply(clean_model_name)

# ২. 'Local' মডেলটিকে ডাটাফ্রেম থেকে সম্পূর্ণ বাদ দেওয়া
df = df[df['model'] != 'Local']

unique_models = sorted(df['model'].unique())
print("Models to be plotted:", unique_models)

# ── Exact Color Map For Models ────────────────────────────────
BASE_COLORS = {
    'QCapsule':   '#2ca02c',  # সবুজ
    'qMPS':       '#9467bd',  # বেগুনী
    'Layer':      '#ff7f0e',  # অরেঞ্জ
    'ResQNet':    '#d62728',  # লাল
    'Global':     '#8c564b',  # খয়েরি
    'Glob96':     '#e377c2',  # গোলাপী
}

# ── Figure Style Layout ───────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'axes.edgecolor':    'black',
    'axes.linewidth':    1.0,
})

# গ্রিড লেআউট তৈরি (সর্বোচ্চ ৪টি কলাম)
n_models = len(unique_models)
n_cols = min(4, n_models)
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.8 * n_rows), squeeze=False)
axes = axes.flatten()

# ── Plotting Grid Mesh for Each Model ─────────────────────────
idx = 0
for model_name in unique_models:
    ax = axes[idx]
    model_df = df[df['model'] == model_name]
    
    x = model_df['theta1'].values
    y = model_df['theta2'].values
    z = model_df['cost'].values
    
    # ডাটা গ্রিড তৈরি করা
    xi = np.linspace(x.min(), x.max(), 100)
    yi = np.linspace(y.min(), y.max(), 100)
    X, Y = np.meshgrid(xi, yi)
    
    # কস্ট ভ্যালুর ডিফারেন্স চেক করে লগ স্কেল হ্যান্ডলিং
    if z.min() > 0 and (z.max() / z.min() > 100):
        Z = griddata((x, y), np.log10(z), (X, Y), method='cubic')
        is_log = True
    else:
        Z = griddata((x, y), z, (X, Y), method='cubic')
        is_log = False

    # কাস্টম লাইট-টু-ডার্ক কালারশেড জেনারেট করা
    base_color = BASE_COLORS.get(model_name, '#333333')
    cmap = LinearSegmentedColormap.from_list(
        'custom_cmap', ['#ffffff', base_color], N=256
    )
    
    # ২D কনট্যুর ফিল্ড জেনারেট করা
    contour = ax.contourf(X, Y, Z, levels=30, cmap=cmap)
    
    # বর্ডার কনট্যুর লাইন
    ax.contour(X, Y, Z, levels=10, colors='black', alpha=0.15, linewidths=0.5)
    
    # কালারবার (Colorbar) যোগ করা
    cbar = fig.colorbar(contour, ax=ax, shrink=0.75, aspect=15)
    cbar.ax.tick_params(labelsize=8)
    if is_log:
        cbar.set_label(r'$\log_{10}(\mathrm{Cost})$', fontsize=9)
    else:
        cbar.set_label('Cost Value', fontsize=9)

    # সেটিংস এবং টাইটেল
    ax.set_title(model_name, fontweight='bold', pad=8)
    ax.set_xlabel(r'$\theta_1$', labelpad=4)
    ax.set_ylabel(r'$\theta_2$', labelpad=4)
    
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min(), y.max())
    
    # অক্ষের ভেতরের দিকে টিক্স রাখা
    ax.tick_params(direction='in', which='both', top=True, right=True)
    idx += 1

# অতিরিক্ত ফাঁকা প্লটগুলো মুছে ফেলা
for j in range(idx, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

# ── Save Plot ─────────────────────────────────────────────────
plt.savefig(OUT_PATH, dpi=300, bbox_inches='tight')
plt.close()

print(f'✓ Beautiful 2D Landscape Grid saved successfully at:\n  {OUT_PATH}')

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

file_path = r"C:\Users\User\Downloads\baseline experiments\exp3_gradient_histogram.csv"
df = pd.read_csv(file_path)

# ── সব 6টা মডেল রাখা ──
models_to_keep = ["QCN", "B5-ResQNet", "B6-Global", "B7-Glob96", "B2-qMPS", "B3-Layer"]
df_filtered = df[df["model"].isin(models_to_keep)].copy()
df_filtered["model"] = df_filtered["model"].replace({"QCN": "QCapsule"})
df_filtered["model"] = df_filtered["model"].replace({"B5-ResQNet": "ResQNet"})
df_filtered["model"] = df_filtered["model"].replace({"B6-Global": "Global"})
df_filtered["model"] = df_filtered["model"].replace({"B7-Glob96": "Glob96"})
df_filtered["model"] = df_filtered["model"].replace({"B2-qMPS": "qMPS"})
df_filtered["model"] = df_filtered["model"].replace({"B3-Layer": "Layer"})

BASE_COLORS = {
    "QCapsule":   "#2ca02c",
    "ResQNet": "#d62728",
    "Global":  "#8c564b",
    "Glob96":  "#e377c2",
    "qMPS":    "#1f77b4",
    "Layer":   "#ff7f0e",
}

order = ["QCapsule", "Layer", "ResQNet", "Glob96", "qMPS", "Global"]

sns.set_theme(style="whitegrid")

# ══════════════════════════════════════════════
# Option A: একটা plot, xlim clipped
# ══════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 6), dpi=300)

for model in order:
    sub = df_filtered[df_filtered["model"] == model]["gradient"]
    sns.kdeplot(
        sub,
        ax=ax,
        color=BASE_COLORS[model],
        label=model,
        linewidth=2,
        fill=True,
        alpha=0.20,
        clip=(-0.5, 0.5),
    )

ax.set_xlim(-0.4, 0.4)
ax.set_ylim(0, 80)

ax.set_title(
    "Gradient Distribution — Clipped View (|∂C| < 0.4)",
    fontsize=13, fontweight="bold", pad=12,
)
ax.set_xlabel(r"Gradient Value ($\partial_\theta C$)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.legend(title="Models", frameon=True, fontsize=9)
plt.tight_layout()
plt.savefig(
    os.path.join(os.path.dirname(file_path), "grad_hist_clipped.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()
print("Saved: grad_hist_clipped.png")


# ══════════════════════════════════════════════
# Option B: 6-subplot (2×3), প্রতিটা নিজের scale
# ══════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300)
axes = axes.flatten()

for i, model in enumerate(order):
    sub = df_filtered[df_filtered["model"] == model]["gradient"]
    ax = axes[i]

    sns.kdeplot(
        sub,
        ax=ax,
        color=BASE_COLORS[model],
        linewidth=2,
        fill=True,
        alpha=0.3,
    )

    ax.set_title(model, fontsize=12, fontweight="bold",
                 color=BASE_COLORS[model])
    ax.set_xlabel(r"$\partial_\theta C$", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)

    # নিজের spread অনুযায়ী xlim
    p1  = np.percentile(sub, 1)
    p99 = np.percentile(sub, 99)
    margin = (p99 - p1) * 0.3
    ax.set_xlim(p1 - margin, p99 + margin)

    # variance + mean annotation
    var  = sub.var()
    mean = sub.mean()
    ax.text(
        0.97, 0.95,
        f"Var  = {var:.2e}\nMean = {mean:.2e}",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=8.5,
        color=BASE_COLORS[model],
        bbox=dict(boxstyle="round,pad=0.3", fc="white",
                  ec=BASE_COLORS[model], alpha=0.75)
    )

    # zero-line reference
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)

fig.suptitle(
    "Gradient Distribution per Architecture",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(os.path.dirname(file_path), "grad_hist_subplots.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()
print("Saved: grad_hist_subplots.png")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path = r"C:\Users\User\Downloads\baseline experiments\exp4_grad_var_depth.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"ফাইল পাওয়া যায়নি: {file_path}")

df = pd.read_csv(file_path)
df['model'] = df['model'].replace({'QCN': 'QCapsule'})

sns.set_theme(style="whitegrid", rc={"grid.linestyle": "--", "grid.alpha": 0.5})
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
})

models = df['model'].unique()
n_models = len(models)

ncols = 3                                          # ৬ মডেল → 2×3 বেশি সুন্দর
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 4 * nrows),
    sharey=False
)
axes = axes.flatten()

qubit_colors = {
    2:  '#1f77b4',
    4:  '#ff7f0e',
    6:  '#d62728',
    8:  '#2ca02c',
    10: '#bcbd22',
    12: '#9467bd',
    14: '#7f7f7f',
    16: '#e377c2',
    18: '#8c564b',
}

for i, model_name in enumerate(models):
    ax = axes[i]
    model_df = df[df['model'] == model_name]
    qubits = sorted(model_df['n_qubits'].unique())

    for q in qubits:
        q_df = model_df[model_df['n_qubits'] == q].sort_values('depth')
        depths = q_df['depth'].values
        means  = q_df['grad_var_mean'].values
        stds   = q_df['grad_var_std'].values
        color  = qubit_colors.get(q, '#000000')

        ax.plot(depths, means, marker='o', markersize=4,
                linewidth=1.5, label=f'{q} qubits', color=color)
        ax.fill_between(
            depths,
            np.maximum(means - stds, 1e-20),   # 1e-10 → 1e-20, log scale safe
            means + stds,
            color=color, alpha=0.1
        )

    ax.set_yscale('log')
    ax.set_title(model_name, weight='bold', color=plt.cm.tab10(i % 10))
    ax.set_xlabel('Depth')
    ax.set_ylabel('Gradient variance')

    unique_depths = sorted(model_df['depth'].unique())
    ax.set_xticks([1, 2, 3, 4, 5, 10, 15, 20, 25]
                  if max(unique_depths) > 5 else unique_depths)

    ax.legend(loc='lower left', fontsize=8, framealpha=0.8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle(
    'Gradient Variance vs Depth — All Models',
    fontsize=14, weight='bold', y=1.01
)
plt.tight_layout()

# ── plt.show() সরিয়ে savefig + close ──
output_path = os.path.join(
    os.path.dirname(file_path), "exp4_grad_var_depth.png"
)
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {output_path}")